In [1]:
# ============================================================
# Progression-Free Survival (PFS) logic:
#
# PFS is calculated at the patient + metastatic therapy line level,
# not simply row-by-row.
#
# Reason:
#   One metastatic therapy line can contain multiple concurrent drugs.
#   For example, patient 1472-17 First-line therapy:
#
#       Met_Hormonal_Therapy   start = 4760
#       Met_Targeted_Therapy   start = 4776
#       Met_Targeted_Therapy   start = 4776
#       End/progression day    = 4965
#
#   The correct treatment-line start is the earliest start date:
#
#       min(4760, 4776, 4776) = 4760
#
#   Therefore:
#
#       PFS = EndTrt_Days_From_Primary_Diagnosis
#             - earliest line start date
#
#       PFS = 4965 - 4760 = 205 days
#
# Implementation:
#   For each patient_id + Met_Therapy_Line:
#       Line_Start_Days_From_Primary_Diagnosis =
#           min(Days_From_Primary_Diagnosis)
#
#   Then, for rows where:
#       Treatment_Response == "Progression"
#
#   Calculate:
#       Progression_Free_Survival =
#           EndTrt_Days_From_Primary_Diagnosis
#           - Line_Start_Days_From_Primary_Diagnosis
#
# Negative PFS handling:
#   If calculated PFS is negative, this likely indicates a source-data issue,
#   for example treatment end/progression day earlier than treatment start day.
#   These values are set to NA in the final PFS column and flagged in
#   PFS_QC_Flag for manual review.
#
#
# Blood-draw TFx logic:
#
# TFx is added at the blood-draw sample level by matching Barcode.
#
# Example:
#   Clinical Barcode:    AMU9-NB1
#   Blood_draws Barcode: AMU9-NB1
#
# Matching key:
#   clinical["Barcode"] == blood_draws["Barcode"]
#
# We match by Barcode, not patient_id, because one patient can have multiple
# blood draws:
#
#   NB1, NB2, NB3, NB4, etc.
#
# Therefore, patient_id alone is not specific enough to assign TFx to the
# correct blood draw.
#
# Only rows with NB blood-draw barcodes receive TFx values. Other clinical
# rows such as Met_Diagnosis, Met_Therapy, Progression_onTx, date_of_death,
# and Overall_Survival_days remain blank for TFx.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# 1. Paths
# ============================================================

BASE_DIR = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA")

clinical_xlsx = (
    BASE_DIR /
    "Clinical_data/AUP_ClinicalData_20260609.xlsx"
)

clinical_sheet = "AUP_ClinicalData_20260609.NO.DA"

blood_draws_xlsx = (
    BASE_DIR /
    "Clinical_data/All_cases_clinical_CNV_extraction_TSVs/"
    "All_cases_clinical_CNV_extraction.xlsx"
)

blood_draws_sheet = "Blood_draws"

out_dir = BASE_DIR / "Survival"
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / "AUP_ClinicalData_20260609_NO_DA_with_PFS_and_blood_draw_TFx.csv"

print("Clinical file:")
print(clinical_xlsx)

print("\nBlood-draw TFx file:")
print(blood_draws_xlsx)

print("\nOutput file:")
print(out_csv)


# ============================================================
# 2. Helper functions
# ============================================================

def clean_patient_id(x):
    """
    Clean patient_id so numeric and character IDs match consistently.

    Examples:
      8       -> "8"
      8.0     -> "8"
      "8.0"   -> "8"
      "1472-17" -> "1472-17"
    """
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()

    if s == "" or s.lower() in {"na", "nan", "none", "null"}:
        return pd.NA

    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass

    return s


def clean_barcode(x):
    """
    Clean barcode for exact matching.

    Examples:
      "AMU9-NB1"    -> "AMU9-NB1"
      " amu9-nb1 "  -> "AMU9-NB1"
    """
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()

    if s == "" or s.lower() in {"na", "nan", "none", "null"}:
        return pd.NA

    return s.upper()


def find_first_col(df, candidates, label):
    """
    Find the first matching column name from a candidate list.
    """
    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        f"Could not find {label}.\n"
        f"Tried: {candidates}\n\n"
        f"Available columns:\n{df.columns.tolist()}"
    )


def safe_numeric(x):
    """
    Convert to numeric. Invalid values become NaN.
    """
    return pd.to_numeric(x, errors="coerce")


def excel_col_to_index(col_letter):
    """
    Convert Excel column letter to zero-based Python index.

    Examples:
      A  -> 0
      AF -> 31
      AG -> 32
      AH -> 33
    """
    col_letter = col_letter.upper()
    index = 0

    for char in col_letter:
        index = index * 26 + (ord(char) - ord("A") + 1)

    return index - 1


def insert_or_move_column(df, col_name, values, excel_col_letter):
    """
    Insert or move a column to a desired Excel-like position.
    """
    if col_name in df.columns:
        df = df.drop(columns=[col_name])

    target_index = excel_col_to_index(excel_col_letter)
    insert_index = min(target_index, len(df.columns))

    df.insert(insert_index, col_name, values)

    return df


# ============================================================
# 3. Read blood-draw TFx data
# ============================================================

blood = pd.read_excel(
    blood_draws_xlsx,
    sheet_name=blood_draws_sheet,
    dtype=str
)

print("\n========== Blood_draws input ==========")
print("Rows:", blood.shape[0])
print("Columns:", blood.shape[1])
print(blood.columns.tolist())

blood_barcode_col = find_first_col(
    blood,
    candidates=[
        "Barcode",
        "barcode",
        "Aurora_barcode",
        "AURORA_barcode",
        "Sample_Barcode",
        "Sample",
        "sample",
        "Sample_ID",
        "sample_id"
    ],
    label="blood-draw barcode column"
)

tfx_col = find_first_col(
    blood,
    candidates=[
        "Tumor_Fraction_percent_from_CNV_annotation",
        "Tumor_Fraction_percent",
        "TFx_percent",
        "Tumor_Fraction",
        "TFx"
    ],
    label="tumor fraction / TFx column"
)

print("\nDetected blood-draw columns:")
print("Barcode column:", blood_barcode_col)
print("TFx column:", tfx_col)

blood["barcode_clean"] = blood[blood_barcode_col].apply(clean_barcode)
blood["TFx_numeric"] = safe_numeric(blood[tfx_col])

# If TFx appears as 0-1 fraction, convert to percent
if blood["TFx_numeric"].dropna().shape[0] > 0:
    if blood["TFx_numeric"].dropna().max() <= 1:
        print("\nTFx appears to be stored as 0-1 fraction. Converting to percent.")
        blood["TFx_numeric"] = blood["TFx_numeric"] * 100

print("\n========== Blood-draw TFx summary ==========")
print("Rows in Blood_draws:", blood.shape[0])
print("Rows with nonmissing barcode:", blood["barcode_clean"].notna().sum())
print("Rows with nonmissing TFx:", blood["TFx_numeric"].notna().sum())
print("Unique barcodes in Blood_draws:", blood["barcode_clean"].nunique())
print(
    "Unique barcodes with nonmissing TFx:",
    blood.loc[blood["TFx_numeric"].notna(), "barcode_clean"].nunique()
)

# One TFx value per barcode.
# If duplicate barcode rows exist, median TFx is used.
blood_tfx_by_barcode = (
    blood.loc[blood["barcode_clean"].notna()]
    .groupby("barcode_clean", dropna=False)
    .agg(
        TFx=("TFx_numeric", "median"),
        n_blood_rows_for_barcode=("TFx_numeric", "size"),
        n_nonmissing_TFx_for_barcode=("TFx_numeric", lambda x: x.notna().sum())
    )
    .reset_index()
)

duplicate_barcodes = blood_tfx_by_barcode[
    blood_tfx_by_barcode["n_blood_rows_for_barcode"] > 1
].copy()

print("\nBlood-draw barcodes with duplicate rows:", duplicate_barcodes.shape[0])

if duplicate_barcodes.shape[0] > 0:
    print("\nPreview duplicate blood-draw barcodes:")
    print(duplicate_barcodes.head(20).to_string(index=False))


# ============================================================
# 4. Read clinical data
# ============================================================

clinical = pd.read_excel(
    clinical_xlsx,
    sheet_name=clinical_sheet,
    dtype=str
)

print("\n========== Clinical input ==========")
print("Rows:", clinical.shape[0])
print("Columns:", clinical.shape[1])
print(clinical.columns.tolist())

patient_col = find_first_col(
    clinical,
    candidates=["patient_id", "Patient_ID", "patientID"],
    label="patient_id column"
)

category_col = find_first_col(
    clinical,
    candidates=["Category"],
    label="Category column"
)

barcode_col = find_first_col(
    clinical,
    candidates=[
        "Barcode",
        "barcode",
        "Aurora_barcode",
        "AURORA_barcode"
    ],
    label="clinical Barcode column"
)

line_col = find_first_col(
    clinical,
    candidates=["Met_Therapy_Line"],
    label="Met_Therapy_Line column"
)

start_col = find_first_col(
    clinical,
    candidates=["Days_From_Primary_Diagnosis"],
    label="Days_From_Primary_Diagnosis column"
)

end_col = find_first_col(
    clinical,
    candidates=["EndTrt_Days_From_Primary_Diagnosis"],
    label="EndTrt_Days_From_Primary_Diagnosis column"
)

response_col = find_first_col(
    clinical,
    candidates=["Treatment_Response"],
    label="Treatment_Response column"
)

clinical["patient_id_clean"] = clinical[patient_col].apply(clean_patient_id)
clinical["barcode_clean"] = clinical[barcode_col].apply(clean_barcode)


# ============================================================
# 5. Calculate Progression-Free Survival
# ============================================================

pfs_col = "Progression_Free_Survival"
pfs_flag_col = "PFS_QC_Flag"
line_start_col = "Line_Start_Days_From_Primary_Diagnosis"

clinical[line_col] = (
    clinical[line_col]
    .astype(str)
    .str.strip()
    .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
)

clinical[category_col] = clinical[category_col].astype(str).str.strip()

start_days = safe_numeric(clinical[start_col])
end_days = safe_numeric(clinical[end_col])

response_clean = (
    clinical[response_col]
    .astype(str)
    .str.strip()
    .str.lower()
)

therapy_row_mask = (
    clinical["patient_id_clean"].notna()
    & clinical[line_col].notna()
    & start_days.notna()
)

print("\n========== Therapy rows for PFS ==========")
print("Rows with usable patient_id + Met_Therapy_Line + start day:", int(therapy_row_mask.sum()))

line_start_df = clinical.loc[
    therapy_row_mask,
    ["patient_id_clean", line_col]
].copy()

line_start_df[line_start_col] = start_days.loc[therapy_row_mask].values

line_start_summary = (
    line_start_df
    .groupby(["patient_id_clean", line_col], dropna=False)
    .agg(
        Line_Start_Days_From_Primary_Diagnosis=(line_start_col, "min"),
        n_rows_in_patient_line=(line_start_col, "size")
    )
    .reset_index()
)

print("Unique patient + therapy-line combinations:", line_start_summary.shape[0])

clinical = clinical.merge(
    line_start_summary,
    on=["patient_id_clean", line_col],
    how="left"
)

line_start_days = safe_numeric(clinical[line_start_col])

pfs_raw = pd.Series(pd.NA, index=clinical.index, dtype="object")
pfs_flag = pd.Series(pd.NA, index=clinical.index, dtype="object")

progression_mask = response_clean.eq("progression")

valid_pfs_mask = (
    progression_mask
    & end_days.notna()
    & line_start_days.notna()
)

pfs_raw.loc[valid_pfs_mask] = (
    end_days.loc[valid_pfs_mask] - line_start_days.loc[valid_pfs_mask]
)

pfs_numeric = safe_numeric(pfs_raw)

negative_pfs_mask = pfs_numeric < 0

pfs_final = pfs_raw.copy()
pfs_final.loc[negative_pfs_mask] = pd.NA

pfs_flag.loc[negative_pfs_mask] = (
    "Source-data issue: treatment end/progression day is earlier than "
    "treatment-line start day; PFS set to NA"
)

print("\n========== PFS calculation summary ==========")
print("Rows with Treatment_Response == Progression:", int(progression_mask.sum()))
print("Rows with raw calculated PFS:", int(valid_pfs_mask.sum()))
print("Rows with negative raw PFS:", int(negative_pfs_mask.sum()))
print("Rows with final nonmissing PFS:", int(safe_numeric(pfs_final).notna().sum()))

if int(negative_pfs_mask.sum()) > 0:
    negative_preview_cols = [
        patient_col,
        "patient_id_clean",
        category_col,
        line_col,
        start_col,
        line_start_col,
        end_col,
        response_col
    ]

    negative_preview = clinical.loc[
        negative_pfs_mask,
        negative_preview_cols
    ].copy()

    negative_preview["Progression_Free_Survival_raw"] = pfs_raw.loc[negative_pfs_mask].values
    negative_preview[pfs_col] = pfs_final.loc[negative_pfs_mask].values
    negative_preview[pfs_flag_col] = pfs_flag.loc[negative_pfs_mask].values

    print("\n========== Negative PFS rows for manual QC ==========")
    print(negative_preview.to_string(index=False))


# ============================================================
# 6. Match blood-draw TFx by Barcode
# ============================================================

clinical_blood_mask = (
    clinical["barcode_clean"]
    .astype("string")
    .str.contains(r"-NB\d+$", case=False, na=False)
)

clinical = clinical.merge(
    blood_tfx_by_barcode,
    on="barcode_clean",
    how="left"
)

clinical_nb = clinical.loc[clinical_blood_mask].copy()

n_clinical_nb_rows = clinical_nb.shape[0]
n_unique_clinical_nb = clinical_nb["barcode_clean"].nunique()

n_matched_nb_rows = clinical_nb["n_blood_rows_for_barcode"].notna().sum()
n_unique_matched_nb = clinical_nb.loc[
    clinical_nb["n_blood_rows_for_barcode"].notna(),
    "barcode_clean"
].nunique()

n_rows_with_tfx = clinical_nb["TFx"].notna().sum()
n_unique_with_tfx = clinical_nb.loc[
    clinical_nb["TFx"].notna(),
    "barcode_clean"
].nunique()

print("\n========== Blood-draw TFx matching summary ==========")
print("Clinical rows with NB blood-draw barcode:", n_clinical_nb_rows)
print("Unique NB blood-draw barcodes in clinical file:", n_unique_clinical_nb)
print("Clinical NB rows matched to Blood_draws sheet:", n_matched_nb_rows)
print("Unique NB barcodes matched to Blood_draws sheet:", n_unique_matched_nb)
print("Clinical NB rows with nonmissing TFx:", n_rows_with_tfx)
print("Unique NB barcodes with nonmissing TFx:", n_unique_with_tfx)

unmatched_nb = (
    clinical_nb.loc[
        clinical_nb["n_blood_rows_for_barcode"].isna(),
        [patient_col, "patient_id_clean", barcode_col, "barcode_clean"]
    ]
    .drop_duplicates()
    .sort_values("barcode_clean")
)

print("\n========== Clinical NB barcodes not matched to Blood_draws ==========")
print("Unmatched unique NB barcodes:", unmatched_nb["barcode_clean"].nunique())

if unmatched_nb.shape[0] > 0:
    print(unmatched_nb.to_string(index=False))


# ============================================================
# 7. Insert final columns at requested positions
# ============================================================
# AF = Progression_Free_Survival
# AG = PFS_QC_Flag
# AH = TFx
#
# Note:
# CSV files do not store Excel column letters internally.
# This controls the column order so that when opened in Excel,
# these columns appear at the intended positions.

clinical = insert_or_move_column(
    clinical,
    col_name=pfs_col,
    values=pfs_final,
    excel_col_letter="AF"
)

clinical = insert_or_move_column(
    clinical,
    col_name=pfs_flag_col,
    values=pfs_flag,
    excel_col_letter="AG"
)

clinical = insert_or_move_column(
    clinical,
    col_name="TFx",
    values=clinical["TFx"],
    excel_col_letter="AH"
)

print("\n========== New column positions ==========")
print(f"{pfs_col} position:", clinical.columns.get_loc(pfs_col) + 1)
print(f"{pfs_flag_col} position:", clinical.columns.get_loc(pfs_flag_col) + 1)
print("TFx position:", clinical.columns.get_loc("TFx") + 1)


# ============================================================
# 8. Optional cleanup of internal helper columns
# ============================================================
# These are useful during QC but usually not needed in final output.
# Keep Line_Start_Days_From_Primary_Diagnosis because it helps verify PFS.

cols_to_drop = [
    "barcode_clean",
    "n_blood_rows_for_barcode",
    "n_nonmissing_TFx_for_barcode",
    "n_rows_in_patient_line"
]

clinical = clinical.drop(
    columns=[c for c in cols_to_drop if c in clinical.columns]
)


# ============================================================
# 9. Save output CSV
# ============================================================

clinical.to_csv(out_csv, index=False)

print("\n========== Saved output ==========")
print(out_csv)

print("\nFinal output dimensions:")
print(clinical.shape)

print("\nPreview of key columns:")
preview_cols = [
    patient_col,
    category_col,
    barcode_col,
    line_col,
    start_col,
    line_start_col,
    end_col,
    response_col,
    pfs_col,
    pfs_flag_col,
    "TFx"
]

preview_cols = [c for c in preview_cols if c in clinical.columns]

print(
    clinical.loc[
        clinical[pfs_col].notna() | clinical["TFx"].notna(),
        preview_cols
    ]
    .head(30)
    .to_string(index=False)
)

Clinical file:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/AUP_ClinicalData_20260609.xlsx

Blood-draw TFx file:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/All_cases_clinical_CNV_extraction_TSVs/All_cases_clinical_CNV_extraction.xlsx

Output file:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Survival/AUP_ClinicalData_20260609_NO_DA_with_PFS_and_blood_draw_TFx.csv

========== Blood_draws input ==========
Rows: 769
Columns: 38
['Patient_Identifier', 'patient_id', 'Site', 'mapping_status', 'Blood_Draw_Days_From_Primary_Diagnosis', 'Barcode', 'Clinical_ID', 'Clinical_ID_2', 'Active_treatment_at_draw', 'Treatment_context_at_draw', 'Most_recent_treatment_before_draw', 'Most_recent_treatment_start_day', 'Most_recent_treatment_end_day', 'Most_recent_treatment_response_recorded', 'Most_recent_treatment_progression_recorded', 'Most_recent_treatment_reason_ended_recorded', 'Most_recent_completed_treatment_before_draw', 'Most_recent_completed_t